# Buyer GPT B3 and B4: what the scores really measure

**Sell Anything, notebook 7.** B4 is B3 trained with one more file of sales, `sales_b3.jsonl` (113 sales, 16 of
them held out for testing): the sales written to the scene quotas (seeds 2500–2592) and the first sellers who write
like real players (2650–2669). On the two test sets of notebook 6, B4 was
not better than B3. This notebook asks why, with measurements suggested by the papers we read
(`docs/paper-ideas.md`):

| Section | Question | Paper idea |
|---|---|---|
| 1 | How were B3 and B4 trained? | |
| 2 | Run the test again on the CPU and keep every line | |
| 3 | A third test set in the new style | |
| 4 | Are the scores high only on lines like the training ones? | B1 |
| 5 | Is the model reciting its training lines? | B2 |
| 6 | Does it repeat itself? | B12 |
| 7 | Does it still read an answer typed the way a player types? | B17 |
| 8 | Which writer's style is hardest to learn? | B19 |
| 9 | Did format 6's whole-word tokens change anything? | B21 |
| 10 | Does our cat market point the same way as real adoptions? | |
| 11 | Half the data: does more data still help? | B15 |
| 12 | What we do next | |

This notebook only reads files the runs saved; it runs no model and opens in seconds.

In [1]:
import hashlib
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

ML = Path.cwd().parent
M = ML / "runs/measure_b1"          # everything measured for this notebook
P = ML / "runs/eval_sets_b2/player"
SETS = {"old": "val_old", "new": "val_new", "new style": "val_b3"}


def load(path):
    return json.loads(Path(path).read_text())


def rows(path):
    return [json.loads(line) for line in open(path)]

## 1. Training B3 and B4

Both start from `pretrain_large` (71.0M parameters) and use the same recipe: 3 epochs, batch 16, lr 3e-4, loss on
the target only, prompt format 5. The only difference is the sales.

In [2]:
for name, data in (("sft_b3_resume", "buyers_b3"), ("sft_b4", "buyers_b4")):
    meta = load(ML / f"data/processed/{data}/meta.json")
    curve = [r for r in rows(ML / f"runs/{name}/metrics.jsonl") if "val_loss" in r]
    best = min(curve, key=lambda r: r["val_loss"])
    print(f"{name}: {meta['sales']['train']} training sales, {sum(meta['examples']['train'].values()):,} examples; "
          f"best val loss {best['val_loss']:.3f} at step {best['step']} of {curve[-1]['step']}")

sft_b3_resume: 323 training sales, 37,812 examples; best val loss 1.293 at step 7089 of 7089
sft_b4: 412 training sales, 48,832 examples; best val loss 1.247 at step 9156 of 9156


B4 trains on 412 sales against B3's 323, and its best validation loss is lower (1.247 against 1.293). The two losses are not comparable, though: B4's validation set includes the held-out sales of
the new batch, so it is a different set. Sections 2 to 9 compare the two models on the same prompts instead.

## 2. The test again, on the CPU, with every line kept

`tools/eval_buyer_b1.py` writes each line the way the game does (temperature 0.7, top-k 50, a fixed seed per
prompt) and checks it with the same program rules the training sales passed. Before, it kept only the scores and 12
examples per task. It now takes `--outputs` and keeps every line with its checks, so the later sections can look
inside the scores. Both models were run on the CPU (the GPU was painting cats; the CPU and the GPU draw different
random numbers, so models compared here are always run on the same device).

In [3]:
for m in ("b3", "b4"):
    for label, s in SETS.items():
        out = rows(M / f"{m}_{s}_outputs.jsonl")
        print(f"{m.upper()} {label:9s}: {len(out):5d} lines kept, tasks {dict(Counter(r['task'] for r in out))}")

B3 old      :  1195 lines kept, tasks {'open': 100, 'ask': 98, 'react': 908, 'imagine': 89}
B3 new      :  1998 lines kept, tasks {'open': 170, 'ask': 163, 'react': 1513, 'imagine': 152}
B3 new style:  1752 lines kept, tasks {'open': 140, 'ask': 140, 'react': 1339, 'imagine': 133}
B4 old      :  1195 lines kept, tasks {'open': 100, 'ask': 98, 'react': 908, 'imagine': 89}
B4 new      :  1998 lines kept, tasks {'open': 170, 'ask': 163, 'react': 1513, 'imagine': 152}
B4 new style:  1752 lines kept, tasks {'open': 140, 'ask': 140, 'react': 1339, 'imagine': 133}


Every prompt of every set has its line kept for both models: 1,195 lines on the old set, 1,998 on the new set and
1,752 on the new-style set (section 3). In every set about nine in ten prompts are reactions.

## 3. A third test set: the new style

B4's extra sales are written in a new style (scene quotas, sellers who type like players). The old and new sets of
notebook 6 have none of it. The new-style set is every held-out sale of `sales_b3.jsonl`: 16 sales (13 from the scene
quotas, 3 with a player-like seller). Neither B3 nor B4 trained on any of them.

In [4]:
print(f"{'set':10s}{'model':6s}{'react rules':>12s}{'same direction':>16s}{'offer error':>13s}{'scene rules':>13s}")
for label, s in SETS.items():
    for m in ("b3", "b4"):
        t = load(M / f"{m}_{s}_cpu.json")["tasks"]
        print(f"{label:10s}{m.upper():6s}{t['react']['rules_ok']:12.3f}{t['react']['same_direction']:16.3f}"
              f"{'$' + format(t['react']['mean_offer_error'], '.2f'):>13s}{t['imagine']['rules_ok']:13.3f}")

set       model  react rules  same direction  offer error  scene rules
old       B3           0.924           0.796        $2.51        0.629
old       B4           0.925           0.776        $2.89        0.494
new       B3           0.964           0.771        $2.79        0.225
new       B4           0.959           0.751        $2.78        0.336
new style B3           0.954           0.744        $2.86        0.273
new style B4           0.975           0.814        $2.42        0.252


On the old and new sets B4 is no better than B3 (direction 0.776 against 0.796, and 0.751 against 0.771). On the
new-style set it is clearly better: the reaction moves the same way as the sale's buyer in 81.4% of the prompts
against 74.4%, the offer is off by $2.42 against $2.86, and 97.5% of the reactions pass every rule against 95.4%.
So B4 did learn from its extra sales; the old test sets just do not contain what it learned.

Two cautions. The new-style sales were written by the same writers in the same days as B4's training sales, so part
of the gain may be "the same writers" rather than "the new style" (section 12 plans a cleaner test). And B4 pays for
it on the old set: its scenes pass the rules in 49.4% of the prompts there against B3's 62.9%.

## 4. Are the scores high only on lines like the training ones? (B1)

phi-1 (paper E2, section 5) found that its score was 81.7% on test problems with a close twin in the training data
and 26.9% on the rest. `tools/measure_outputs.py` does the same for us: for each test prompt it finds the training
line of the same task with the most word pairs in common with the sale's own line (bigram F1, like Rouge-2), and
splits the prompts at the median into "near" and "far". The near/far split uses B4's training lines for both models,
so the two halves are the same prompts.

In [5]:
for label, s in SETS.items():
    for m in ("b3", "b4"):
        r = load(M / f"{m}_{s}_measure.json")["tasks"]["react"]["B1"]
        print(f"{label:9s} {m.upper()}: reaction direction near {r['near']['same_direction']:.3f} / far "
              f"{r['far']['same_direction']:.3f}; offer error near ${r['near']['offer_error']:.2f} / far "
              f"${r['far']['offer_error']:.2f}  (cut at bigram F1 {r['median_teacher_f1']})")

old       B3: reaction direction near 0.890 / far 0.702; offer error near $1.40 / far $3.61  (cut at bigram F1 0.607)
old       B4: reaction direction near 0.879 / far 0.674; offer error near $1.50 / far $4.28  (cut at bigram F1 0.607)
new       B3: reaction direction near 0.755 / far 0.784; offer error near $3.75 / far $1.95  (cut at bigram F1 0.333)
new       B4: reaction direction near 0.763 / far 0.741; offer error near $3.09 / far $2.50  (cut at bigram F1 0.333)
new style B3: reaction direction near 0.760 / far 0.727; offer error near $3.13 / far $2.59  (cut at bigram F1 0.359)
new style B4: reaction direction near 0.828 / far 0.800; offer error near $2.58 / far $2.25  (cut at bigram F1 0.359)


On the old set the scores lean on the near half: B3 moves the right way on 89.0% of the near prompts and 70.2% of the
far ones, and its offer is off by $1.40 against $3.61. The old set comes mostly from the Qwen sales, whose lines
repeat each other a lot, so its total score is pulled up by prompts that are nearly in the training data. On the
new and new-style sets the two halves are close (B3 new: 75.5% near, 78.4% far), so those scores are the more honest
ones.

## 5. Is the model reciting? (B2)

TinyStories (paper E1, section 4.4) checked that its model's stories were not copied from the training data. For each
task we count the share of the model's 5-grams that occur anywhere in the training lines, and the share of its lines
that are word for word a training line. The same numbers for the sale's own (held-out) lines are the baseline: a
model that copies more than new human-checked lines do is reciting.

In [6]:
TASKS = ("open", "ask", "react", "imagine")
for label, s in SETS.items():
    for m in ("b3", "b4"):
        r = load(M / f"{m}_{s}_measure.json")["tasks"]
        model = "  ".join(f"{t} {r[t]['B2']['model']['exact_copy']:.2f}/{r[t]['B2']['model']['5gram_in_train']:.2f}" for t in TASKS)
        print(f"{label:9s} {m.upper()} model  : {model}")
    teacher = "  ".join(f"{t} {r[t]['B2']['teacher']['exact_copy']:.2f}/{r[t]['B2']['teacher']['5gram_in_train']:.2f}" for t in TASKS)
    print(f"{label:9s} sale's lines: {teacher}   (exact copies / 5-grams seen in training)")

old       B3 model  : open 0.36/0.71  ask 0.14/0.59  react 0.22/0.44  imagine 0.18/0.59
old       B4 model  : open 0.35/0.66  ask 0.08/0.49  react 0.20/0.45  imagine 0.07/0.55
old       sale's lines: open 0.09/0.44  ask 0.04/0.31  react 0.13/0.40  imagine 0.08/0.48   (exact copies / 5-grams seen in training)
new       B3 model  : open 0.02/0.47  ask 0.07/0.62  react 0.00/0.35  imagine 0.00/0.19
new       B4 model  : open 0.00/0.45  ask 0.07/0.63  react 0.00/0.37  imagine 0.00/0.19
new       sale's lines: open 0.00/0.24  ask 0.00/0.47  react 0.00/0.22  imagine 0.00/0.12   (exact copies / 5-grams seen in training)
new style B3 model  : open 0.04/0.51  ask 0.06/0.59  react 0.00/0.35  imagine 0.00/0.19
new style B4 model  : open 0.01/0.40  ask 0.04/0.53  react 0.00/0.39  imagine 0.00/0.10
new style sale's lines: open 0.00/0.07  ask 0.00/0.42  react 0.00/0.27  imagine 0.00/0.05   (exact copies / 5-grams seen in training)


On the old set the greetings are recited: 36% of B3's and 35% of B4's openings are word for word a training opening,
against 9% of the sale's own. On the new and new-style sets almost nothing is copied whole (at most 7% of the
questions), but about twice as many of the model's 5-grams are training 5-grams as the sale's own (openings on the
new set: 0.47 against 0.24). So the model does not copy lines, but it leans on the training data's set phrases.

## 6. Does it repeat itself? (B12)

He et al. 2018 (paper B1, section 4.4) showed a negotiation model that only says a few lines: its 3 most common lines
were 81.6% to 100% of all it said. We count the same share per task, and distinct-2 (different word pairs over all
word pairs).

In [7]:
for label, s in SETS.items():
    for m in ("b3", "b4"):
        r = load(M / f"{m}_{s}_measure.json")["tasks"]
        print(f"{label:9s} {m.upper()}: " + "  ".join(
            f"{t} {r[t]['B12']['model']['top3_share']:.3f}/{r[t]['B12']['model']['distinct_2']:.2f}" for t in TASKS))
    print(f"{label:9s} sale: " + "  ".join(
        f"{t} {r[t]['B12']['teacher']['top3_share']:.3f}/{r[t]['B12']['teacher']['distinct_2']:.2f}" for t in TASKS)
        + "   (top-3 share / distinct-2)")

old       B3: open 0.060/0.47  ask 0.061/0.48  react 0.033/0.31  imagine 0.067/0.47
old       B4: open 0.080/0.49  ask 0.031/0.49  react 0.036/0.32  imagine 0.045/0.48
old       sale: open 0.030/0.59  ask 0.031/0.59  react 0.039/0.34  imagine 0.034/0.56   (top-3 share / distinct-2)
new       B3: open 0.018/0.55  ask 0.025/0.46  react 0.002/0.35  imagine 0.020/0.57
new       B4: open 0.018/0.59  ask 0.031/0.45  react 0.003/0.35  imagine 0.020/0.57
new       sale: open 0.018/0.70  ask 0.018/0.55  react 0.002/0.47  imagine 0.020/0.68   (top-3 share / distinct-2)
new style B3: open 0.021/0.60  ask 0.043/0.50  react 0.002/0.37  imagine 0.023/0.56
new style B4: open 0.021/0.62  ask 0.029/0.54  react 0.002/0.35  imagine 0.023/0.61
new style sale: open 0.021/0.76  ask 0.021/0.61  react 0.002/0.42  imagine 0.023/0.70   (top-3 share / distinct-2)


Nothing like that collapse: the 3 most common lines are at most 8% of a task's lines, close to the sale's own. The
wording is less varied than the teacher's, though: distinct-2 is about 0.1 lower (new set openings 0.55 for B3 and
0.59 for B4 against 0.70). This goes with section 5: the model reuses set phrases.

## 7. An answer typed the way a player types (B17)

In play, B3 read "20 years old" as 2 and took "heart attack" as good news. `tools/player_style_test.py` takes 55
seller answers from the held-out new set and rewrites each the way a player types (lower case, short forms, 15
typos, "but/though", numbers in digits), keeping every fact, negation and number (checked by a program and read by
hand). The 517 reactions to those answers are run twice with the same seed, once with each wording, so the only
difference is how the answer is typed.

In [8]:
picked = load(P / "picked.json")
for p in [p for p in picked if not p["unchanged"]][:3]:
    print("original:", p["original"]); print("player  :", p["player"]); print()
for m in ("b3", "b4"):
    print(m.upper(), load(P / f"{m}_player_compare.json"))

original: She's a two month old black domestic shorthair, and that coat shines like wet ink. No papers that I know of, but she carries herself like a champion already.
player  : shes 2 months, black domestic shorthair. no papers that i know of but she acts like a lil champion already

original: I can't honestly say she's spent time with children. But she's a playful little thing who's always ready for a game, and kids love that.
player  : cant honstly say shes been around kids. but shes super playful n always up for a game, kids love that

original: She hasn't been tested for either yet, but just look at her bouncing around, she's bursting with energy. One quick test at your vet would settle it in no time.
player  : not tested for either yet but shes bouncing around full of energy. 1 quick test at ur vet would settle it

B3 {'pairs': 516, 'of': 517, 'same_move_original_vs_player': 0.895, 'same_direction_as_sale': {'original': 0.787, 'player': 0.767}, 'rules_ok': {'original': 0.932, 'pl

The player wording costs little here: B3 makes the same move on both wordings for 89.5% of the reactions and B4 for
92.6%; the share moving like the sale's buyer drops by 2.0 points for B3 and 0.9 for B4. This does not show that the
models read players well. The rewrites keep the original's order and facts, and none of these cats has a rare fact
like "20 years" or a heart problem, which is where B3 failed in play. Part of the 7–10% that changed is also just
sampling: we have not yet measured how often the move changes when nothing but the seed changes.

## 8. Which writer's style is hardest to learn? (B19)

Distillation scaling laws (paper E13, section 4.2) found that what decides a student's quality is how hard the
teacher's outputs are to imitate. `tools/val_loss_by_group.py` computes the exact validation loss over all 6,156
validation examples, grouped by who wrote the sale.

In [9]:
b3 = load(M / "b3_loss_by_source.json")["groups"]
b4 = load(M / "b4_loss_by_source.json")["groups"]
for title, keys in (("written by", [k for k in b3 if k.endswith("/ all")]), ("task", [k for k in b3 if k.startswith("all /")])):
    print(f"{title:20s}{'examples':>9s}{'B3 loss':>9s}{'B4 loss':>9s}")
    for key in sorted(keys, key=lambda k: b4[k]["loss_per_token"]):
        print(f"{key:20s}{b3[key]['examples']:9d}{b3[key]['loss_per_token']:9.3f}{b4[key]['loss_per_token']:9.3f}")
    print()

written by           examples  B3 loss  B4 loss
qwen_v9 / all             732    0.808    0.804
claude_b1 / all           463    0.853    0.829
muse / all               1211    1.319    1.276
all / all                6156    1.457    1.284
scene_quota / all        1632    1.830    1.345
format5 / all            1998    1.470    1.413
human_seller / all        120    2.043    1.631

task                 examples  B3 loss  B4 loss
all / react              4682    1.277    1.159
all / all                6156    1.457    1.284
all / ask                 498    1.440    1.312
all / imagine             466    2.019    1.625
all / open                510    2.461    2.020



The tidiest writing is the easiest: the Qwen sales (0.80) and the first Claude sales (0.83) barely change from B3 to
B4. B4 gains where its extra sales are: the scene-quota sales fall from 1.83 to 1.35 and the player-like sellers from
2.04 to 1.63, which agrees with section 3. The player-like sellers stay the hardest even for B4, which trained on
only 17 of them. By task, openings (2.02) and scenes (1.63) are the hardest for B4 and reactions the easiest (1.16).

## 9. Did format 6's whole-word tokens change anything? (B21)

nanochat (paper E12) always reports bits per byte, because a loss per token depends on how much text a token holds.
B2f6 (format 6) gives every drawable thing and every persona a token of its own; notebook 6 compared its loss per
token with B2's (1.334 against 1.316). Here both are scored on the same 3,193 prompts and divided by the bytes of the
same format-5 text.

In [10]:
a, b = load(M / "b2_bpb.json")["groups"], load(M / "b2f6_bpb.json")["groups"]
for key in ("all / all", "all / open", "all / ask", "all / react", "all / imagine"):
    print(f"{key:14s} per token B2 {a[key]['loss_per_token']:.4f}  B2f6 {b[key]['loss_per_token']:.4f}   "
          f"bits per byte B2 {a[key]['bits_per_byte']:.4f}  B2f6 {b[key]['bits_per_byte']:.4f}")

all / all      per token B2 1.3075  B2f6 1.3238   bits per byte B2 0.6376  B2f6 0.6379
all / open     per token B2 1.7317  B2f6 1.7408   bits per byte B2 0.8057  B2f6 0.8100
all / ask      per token B2 1.3967  B2f6 1.4072   bits per byte B2 0.5732  B2f6 0.5775
all / react    per token B2 1.2089  B2f6 1.2067   bits per byte B2 0.6079  B2f6 0.6068
all / imagine  per token B2 1.6854  B2f6 1.9118   bits per byte B2 0.7454  B2f6 0.7509


Per byte the two are the same: 0.6376 against 0.6379. The gap per token (1.3075 against 1.3238 here) comes from the
tokens holding different amounts of text, most of all in the scenes (1.69 against 1.91 per token, but 0.745 against
0.751 per byte). Format 6's tokens neither helped nor hurt the prediction of the text.

## 10. Does our cat market point the same way as real adoptions?

PetFinder.my Adoption Prediction (Kaggle, 2019) has real listings with how fast each pet was adopted. We use it only
for this analysis, under the competition's rules (non-commercial research, not redistributed): the file stays out of
git and is never part of training data or the game. Only cats listed alone are used (4,943), and "not adopted within
100 days" is compared two ways: over all cats, and only between cats of the same rescuer (the 2nd-place solution
grouped its validation by rescuer for the same reason: one rescuer's listings share a writing style and a level of
care). `tools/petfinder_market_check.py`; the write-up is `docs/quality/petfinder-market-check.md`.

In [11]:
csv = ML.parent / "data/external/petfinder/train.csv"
if csv.exists():
    out = subprocess.run([sys.executable, "-I", str(ML / "tools/petfinder_market_check.py"), str(csv)],
                         capture_output=True, text=True).stdout
    print(out)
else:
    print("data/external/petfinder/train.csv is not on this machine (it is never committed); see the write-up.")

4943 single-cat listings from 2115 rescuers
          size   mean
age                  
a0_2m     1631  0.107
a3_5m     1509  0.209
a6_11m     720  0.354
a1_2y      850  0.431
a3y_plus   233  0.429

feature            all (rescuer-clustered)        within rescuer
age_a1_2y             5.93  p=0.000       10.38  p=0.000
age_a3y_plus          6.73  p=0.000       10.96  p=0.000
vacc_yes              1.08  p=0.583        0.83  p=0.435
ster_yes              1.35  p=0.020        1.68  p=0.032
dew_yes               0.64  p=0.001        0.85  p=0.512
injured               1.45  p=0.043        1.10  p=0.840
fur_long              0.30  p=0.000        0.31  p=0.003
free                  0.63  p=0.000        0.69  p=0.197



Only two features hold when the rescuer is held fixed: age (a cat of 1 to 2 years has 10.4 times a 0 to 2 month
kitten's odds of waiting 100 days, 3 years and over 11.0) and long fur (0.31). Dewormed, injured and free lose their
effect within rescuer, so those were the rescuer, not the cat; sterilised even points the opposite way of what a
buyer saves, for a reason the data cannot show. Our asking prices (`tools/asking_price.py`) already fall with age
and price the long-haired breeds up, so they were kept: the game only needs a reasonable market, not PetFinder's.
Injury, deworming, coat and shedding became personal tastes on some buyers' cards instead, so two buyers can take the
same fact opposite ways.

## 11. Half the data: does more data still help? (B15)

Muennighoff et al. (paper E9) found that in pre-training, new data is worth more than repeats, but their appendix
says fine-tuning was not studied. B4-half is B4's recipe on half of its training sales (half of each writer's sales,
`tools/make_half_set.py`), the same validation set and the same number of epochs.

In [12]:
for name in ("sft_b4", "sft_b4_half"):
    curve = [r for r in rows(ML / f"runs/{name}/metrics.jsonl") if "val_loss" in r]
    best = min(curve, key=lambda r: r["val_loss"])
    print(f"{name:12s} best val loss {best['val_loss']:.3f} at step {best['step']}, last {curve[-1]['val_loss']:.3f}")
print()
print(f"{'set':10s}{'model':8s}{'react rules':>12s}{'same direction':>16s}{'offer error':>13s}{'scene rules':>13s}")
for label, s in SETS.items():
    for m in ("b3", "b4half", "b4"):
        t = load(M / f"{m}_{s}_cpu.json")["tasks"]
        print(f"{label:10s}{m.upper():8s}{t['react']['rules_ok']:12.3f}{t['react']['same_direction']:16.3f}"
              f"{'$' + format(t['react']['mean_offer_error'], '.2f'):>13s}{t['imagine']['rules_ok']:13.3f}")

sft_b4       best val loss 1.247 at step 9156, last 1.247
sft_b4_half  best val loss 1.381 at step 3000, last 1.405

set       model    react rules  same direction  offer error  scene rules
old       B3             0.924           0.796        $2.51        0.629
old       B4HALF         0.870           0.713        $3.77        0.303
old       B4             0.925           0.776        $2.89        0.494
new       B3             0.964           0.771        $2.79        0.225
new       B4HALF         0.949           0.699        $3.67        0.211
new       B4             0.959           0.751        $2.78        0.336
new style B3             0.954           0.744        $2.86        0.273
new style B4HALF         0.951           0.742        $3.32        0.227
new style B4             0.975           0.814        $2.42        0.252


B4-half's validation loss is lowest at step 3,000 (1.381) and then rises to 1.405: on half the sales it starts to
overfit, while B4 kept improving to 1.247. On the three test sets, doubling the sales (206 to 412, the same mix of
writers) improves every reaction score: the share moving like the sale's buyer goes from 71.3% to 77.6% on the old
set, 69.9% to 75.1% on the new set and 74.2% to 81.4% on the new-style set, and the offer error falls by about $0.90
on each. B4-half is also below B3 on the old and new sets, though B3 trained on fewer new-style sales and more of the
old ones, so that comparison mixes amount and style.

So the answer to B15 is yes: at a few hundred sales, more sales still clearly help, and writing more is worth it.

**Is the gain just recall?** The test sales are new to both models, but a model that recalls more training lines
can still score higher when a test line has a close twin in training (section 4). Two checks: the near/far split of
section 4 (cut on B4's training lines, so both models face the same halves), and the copying of section 5, each
model against its own training lines.

In [13]:
for label, s in SETS.items():
    for m in ("b4half", "b4"):
        r = load(M / f"{m}_{s}_measure.json")["tasks"]["react"]["B1"]
        print(f"{label:9s} {m.upper():6s} direction near {r['near']['same_direction']:.3f} far {r['far']['same_direction']:.3f}"
              f"   offer error near ${r['near']['offer_error']:.2f} far ${r['far']['offer_error']:.2f}")
print()
for label, s in (("new", "val_new"), ("new style", "val_b3")):
    for m, f in (("b4half", f"b4half_{s}_measure_owntrain.json"), ("b4", f"b4_{s}_measure.json")):
        r = load(M / f)["tasks"]["react"]["B2"]
        print(f"{label:9s} {m.upper():6s} reactions copied whole {r['model']['exact_copy']:.2f}, 5-grams seen in its training "
              f"{r['model']['5gram_in_train']:.2f} against the sale's own {r['teacher']['5gram_in_train']:.2f} "
              f"({r['model']['5gram_in_train'] / r['teacher']['5gram_in_train']:.2f}x)")

old       B4HALF direction near 0.804 far 0.621   offer error near $2.63 far $4.91
old       B4     direction near 0.879 far 0.674   offer error near $1.50 far $4.28
new       B4HALF direction near 0.684 far 0.713   offer error near $4.54 far $2.91
new       B4     direction near 0.763 far 0.741   offer error near $3.09 far $2.50
new style B4HALF direction near 0.744 far 0.740   offer error near $3.45 far $3.18
new style B4     direction near 0.828 far 0.800   offer error near $2.58 far $2.25

new       B4HALF reactions copied whole 0.00, 5-grams seen in its training 0.26 against the sale's own 0.17 (1.54x)
new       B4     reactions copied whole 0.00, 5-grams seen in its training 0.37 against the sale's own 0.22 (1.64x)
new style B4HALF reactions copied whole 0.00, 5-grams seen in its training 0.27 against the sale's own 0.21 (1.26x)
new style B4     reactions copied whole 0.00, 5-grams seen in its training 0.39 against the sale's own 0.27 (1.43x)


Part of the gain is recall: on the near half the share moving like the sale's buyer rises by 7.5 to 8.4 points. But
the far half, where no training line is close, gains too: 62.1% to 67.4% on the old set, 71.3% to 74.1% on the new
set and 74.0% to 80.0% on the new-style set, with a smaller offer error on each. Neither model copies a reaction
whole on the new sets (0.00), and B4 uses the training data's 5-grams only a little more than B4-half relative to
the sale's own lines (1.64x against 1.54x on the new set). Set phrases change the wording, not whether a move is
right on a sale the model has never seen.

One thing these checks cannot rule out: the new-style test sales were written by the same writers in the same days
as B4's extra sales, so their habits (which fact tends to bring which reaction) may be shared even when the words are
not. A whole batch kept out of training (2690–2699) will test that.

## 12. What we do next

| Found | Next |
|---|---|
| B4 gains only in the new style; the player-like sellers are still the hardest (sections 3, 8) | Write the 87 sales left (2593–2649, 2670–2699), with what real adopters ask (origin, food, litter, being held, fleas, microchip, coat) and buyers' personal tastes |
| The new-style gain may partly be "the same writers" (section 3) | Keep one whole batch out of training as a clean test, the way the 2nd PetFinder solution kept whole rescuers apart |
| Half the sales is clearly worse on every set (section 11) | Keep writing sales; the old Qwen sales are easy and recited (sections 4, 5, 8), so new ones should be in the new style |
| Openings and scenes are the hardest tasks; wording less varied (sections 6, 8) | More varied openings and scene sentences in the next sales |
| The player test is mild and has no noise baseline (section 7) | Rerun with only the seed changed; add answers with rare facts ("20 years", heart problems) |
| Format 6 changes nothing per byte (section 9) | Stay with format 5 (and its seller twin, format 8) |

## Version info

The files this notebook read, so the numbers can be traced back.

In [14]:
for f in ["runs/sft_b3_resume/best.pt", "runs/sft_b4/best.pt", "runs/sft_b4_half/best.pt", "data/processed/buyers_b4/train.jsonl",
          "data/processed/buyers_b4/val.jsonl", "runs/eval_sets_b2/val_old_format5.jsonl",
          "runs/eval_sets_b2/val_new_format5.jsonl", "runs/eval_sets_b2/val_b3_format5.jsonl",
          "runs/eval_sets_b2/player/original.jsonl", "runs/eval_sets_b2/player/player.jsonl"]:
    print(hashlib.sha256((ML / f).read_bytes()).hexdigest()[:16], f)

0457c1e04aa247be runs/sft_b3_resume/best.pt


260d6940c00aec74 runs/sft_b4/best.pt


cc384a729d6fedae runs/sft_b4_half/best.pt
b738e4484b798c2f data/processed/buyers_b4/train.jsonl
b7ee1c0a3ff5879e data/processed/buyers_b4/val.jsonl
3e52ea5848e29038 runs/eval_sets_b2/val_old_format5.jsonl
c319c04a9beb8736 runs/eval_sets_b2/val_new_format5.jsonl
0dc11335c2c0bff1 runs/eval_sets_b2/val_b3_format5.jsonl
0d543b87c616c526 runs/eval_sets_b2/player/original.jsonl
0c81863b04159cb5 runs/eval_sets_b2/player/player.jsonl


These hashes identify the three models and the four test sets. All measurements were run on 2026-10-13 and 14 on the
CPU of this Mac.